# 1. Đọc tập Train và Validation

In [1]:
from pathlib import Path

import pandas as pd
import numpy as np

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import (
    OneHotEncoder,
    StandardScaler
)
from sklearn.base import clone

# Xác định thư mục gốc IoT_final
current = Path.cwd().resolve()

ROOT = next(
    (
        folder
        for folder in [current, *current.parents]
        if (folder / "project").is_dir()
        and (folder / "result").is_dir()
    ),
    None
)

if ROOT is None:
    raise FileNotFoundError("Không tìm thấy IoT_final")

# Dữ liệu đã chia ở C02
DATA_DIR = (
    ROOT / "result" / "preprocessing"
    / "C02_Data_Splitting_ThanhDai"
    / "data" / "processed"
)

TRAIN_PATH = DATA_DIR / "train.csv"
VALID_PATH = DATA_DIR / "validation.csv"

# Thư mục kết quả P01
RESULT_DIR = (
    ROOT / "result" / "preprocessing"
    / "P01_MLP_Preprocessing_Baseline"
)

TABLE_DIR = RESULT_DIR / "tables"
TABLE_DIR.mkdir(parents=True, exist_ok=True)

# Đọc hai tập
train_df = pd.read_csv(TRAIN_PATH)
valid_df = pd.read_csv(VALID_PATH)

print("=== THÔNG TIN DỮ LIỆU ===")
print("Train:", train_df.shape)
print("Validation:", valid_df.shape)

print("\n=== CÁC CỘT BAN ĐẦU ===")
display(train_df.head())

assert "Attack_type" in train_df.columns
assert "Attack_type" in valid_df.columns

assert train_df.columns.equals(valid_df.columns)

print("\nĐọc Train và Validation thành công!")

=== THÔNG TIN DỮ LIỆU ===
Train: (82536, 83)
Validation: (17687, 83)

=== CÁC CỘT BAN ĐẦU ===


,id.orig_p,id.resp_p,proto,service,flow_duration,fwd_pkts_tot,bwd_pkts_tot,fwd_data_pkts_tot,bwd_data_pkts_tot,fwd_pkts_per_sec,...,active.std,idle.min,idle.max,idle.tot,idle.avg,idle.std,fwd_init_window_size,bwd_init_window_size,fwd_last_window_size,Attack_type
0,64177,46532,udp,-,0.000000,1,0,1,0,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0,0,0,NMAP_UDP_SCAN
1,20209,21,tcp,-,0.000005,1,1,1,0,199728.761905,...,0.0,0.0,0.0,0.0,0.0,0.0,64,0,64,DOS_SYN_Hping
2,3694,21,tcp,-,0.000005,1,1,1,0,199728.761905,...,0.0,0.0,0.0,0.0,0.0,0.0,64,0,64,DOS_SYN_Hping
3,39875,21,tcp,-,0.000002,1,1,1,0,466033.777778,...,0.0,0.0,0.0,0.0,0.0,0.0,64,0,64,DOS_SYN_Hping
4,9369,21,tcp,-,0.000004,1,1,1,0,246723.764706,...,0.0,0.0,0.0,0.0,0.0,0.0,64,0,64,DOS_SYN_Hping



Đọc Train và Validation thành công!


### Nhận xét: Dữ liệu đầu vào Preprocessing

**Kết quả:**

- Đã đọc hai tập Train và Validation từ kết quả C02.
- Train chiếm khoảng 70% dữ liệu sau Cleaning.
- Validation chiếm khoảng 15%.
- Hai tập có cùng danh sách thuộc tính.
- Nhãn mục tiêu là `Attack_type`.

**Kết luận:**

Dữ liệu đã được làm sạch và chia tập trước khi
thực hiện tiền xử lý.

Các phép biến đổi có tham số học từ dữ liệu
phải được fit trên Train và áp dụng lên Validation.

# 2. Phân loại thuộc tính đầu vào

In [2]:
TARGET = "Attack_type"

# Tách đặc trưng và nhãn
X_train = train_df.drop(columns=[TARGET])
y_train = train_df[TARGET]

X_valid = valid_df.drop(columns=[TARGET])
y_valid = valid_df[TARGET]

# Các đặc trưng phân loại
categorical_features = ["proto", "service"]

# Các đặc trưng số
numeric_features = X_train.select_dtypes(
    include=np.number
).columns.tolist()

# Kiểm tra danh sách đặc trưng
assert all(
    col in X_train.columns
    for col in categorical_features
)

assert set(numeric_features + categorical_features) == set(
    X_train.columns
)

print("=== PHÂN LOẠI ĐẶC TRƯNG ===")

print("Số đặc trưng đầu vào:", X_train.shape[1])
print("Số đặc trưng số:", len(numeric_features))
print(
    "Số đặc trưng phân loại:",
    len(categorical_features)
)

print("\nCác đặc trưng phân loại:")
print(categorical_features)

print("\n=== CÁC GIÁ TRỊ PHÂN LOẠI ===")

for col in categorical_features:
    print(f"\n{col}:")
    display(X_train[col].value_counts())

# Kiểm tra dữ liệu thiếu và vô hạn
assert X_train.isna().sum().sum() == 0
assert X_valid.isna().sum().sum() == 0

assert np.isfinite(
    X_train[numeric_features].to_numpy()
).all()

assert np.isfinite(
    X_valid[numeric_features].to_numpy()
).all()

print("\nKiểm tra đặc trưng hoàn tất!")

=== PHÂN LOẠI ĐẶC TRƯNG ===
Số đặc trưng đầu vào: 82
Số đặc trưng số: 80
Số đặc trưng phân loại: 2

Các đặc trưng phân loại:
['proto', 'service']

=== CÁC GIÁ TRỊ PHÂN LOẠI ===

proto:


proto
tcp     73875
udp      8634
icmp       27
Name: count, dtype: int64


service:


service
-       68755
dns      6628
mqtt     2891
http     2297
ssl      1831
ntp        82
irc        20
ssh        19
dhcp       13
Name: count, dtype: int64


Kiểm tra đặc trưng hoàn tất!


### Nhận xét: Phân loại đặc trưng

**Kết quả:**

- Đã tách nhãn `Attack_type` khỏi tập đầu vào.
- Hai đặc trưng phân loại là `proto` và `service`.
- Các đặc trưng còn lại thuộc dạng số.
- Không phát hiện dữ liệu thiếu hoặc vô hạn
  trong các tập đang sử dụng.
  
**Kết luận:**

Sử dụng One-Hot Encoding cho các đặc trưng
phân loại và StandardScaler cho các đặc trưng số.

Chỉ học tham số tiền xử lý trên Train.

# 3. Xây dựng tiền xử lý

In [3]:
import inspect

def build_preprocessor():

    # Tương thích với các phiên bản scikit-learn
    ohe_params = {
        "handle_unknown": "ignore"
    }

    if "sparse_output" in inspect.signature(
        OneHotEncoder
    ).parameters:
        ohe_params["sparse_output"] = False
    else:
        ohe_params["sparse"] = False

    encoder = OneHotEncoder(**ohe_params)

    transformer = ColumnTransformer(
        transformers=[
            (
                "numeric",
                StandardScaler(),
                numeric_features
            ),
            (
                "categorical",
                encoder,
                categorical_features
            )
        ],
        remainder="drop",
        sparse_threshold=0
    )

    return transformer


# Chưa fit trên bất kỳ dữ liệu nào
preprocessor = build_preprocessor()

print("=== CẤU HÌNH TIỀN XỬ LÝ ===")
print(preprocessor)

=== CẤU HÌNH TIỀN XỬ LÝ ===
ColumnTransformer(sparse_threshold=0,
                  transformers=[('numeric', StandardScaler(),
                                 ['id.orig_p', 'id.resp_p', 'flow_duration',
                                  'fwd_pkts_tot', 'bwd_pkts_tot',
                                  'fwd_data_pkts_tot', 'bwd_data_pkts_tot',
                                  'fwd_pkts_per_sec', 'bwd_pkts_per_sec',
                                  'flow_pkts_per_sec', 'down_up_ratio',
                                  'fwd_header_size_tot', 'fwd_header_size_min',
                                  'fwd_header_size_max', 'bwd_header_size_tot...
                                  'flow_FIN_flag_count', 'flow_SYN_flag_count',
                                  'flow_RST_flag_count', 'fwd_PSH_flag_count',
                                  'bwd_PSH_flag_count', 'flow_ACK_flag_count',
                                  'fwd_URG_flag_count', 'flow_CWR_flag_count',
                             

### Nhận xét: Xây dựng Preprocessing Pipeline

**Phương pháp sử dụng:**

- StandardScaler cho các đặc trưng số.
- One-Hot Encoding cho `proto` và `service`.
- Không đưa `Attack_type` vào quá trình biến đổi.
- Không thực hiện Oversampling hoặc Feature Selection.

**Ý nghĩa:**

StandardScaler giúp chuẩn hóa thang đo của các
đặc trưng số dựa trên trung bình và độ lệch
chuẩn học được từ Train.

One-Hot Encoding giúp chuyển các giá trị phân
loại thành các cột nhị phân mà không áp đặt
thứ tự giả giữa các giá trị.

**Kết luận:**

Đã xây dựng cấu hình tiền xử lý cơ bản cho MLP.

# 4. Kiểm tra khả năng biến đổi Train và Validation

In [4]:
# Tạo bản riêng để kiểm thử
preview_preprocessor = clone(preprocessor)

# Chỉ fit trên Train
X_train_preview = preview_preprocessor.fit_transform(
    X_train
)

# Validation chỉ được transform
X_valid_preview = preview_preprocessor.transform(
    X_valid
)

print("=== KÍCH THƯỚC TRƯỚC TIỀN XỬ LÝ ===")
print("Train:", X_train.shape)
print("Validation:", X_valid.shape)

print("\n=== KÍCH THƯỚC SAU TIỀN XỬ LÝ ===")
print("Train:", X_train_preview.shape)
print("Validation:", X_valid_preview.shape)

# Lấy tên đặc trưng sau One-Hot Encoding
feature_names = (
    preview_preprocessor.get_feature_names_out()
)

print("\nSố đặc trưng sau tiền xử lý:", len(feature_names))

print("\n=== KIỂM TRA GIÁ TRỊ ===")

print(
    "Train có NaN/Inf:",
    not np.isfinite(X_train_preview).all()
)

print(
    "Validation có NaN/Inf:",
    not np.isfinite(X_valid_preview).all()
)

assert X_train_preview.shape[1] == X_valid_preview.shape[1]
assert np.isfinite(X_train_preview).all()
assert np.isfinite(X_valid_preview).all()

# Kiểm tra trung bình và độ lệch chuẩn các cột số
n_numeric = len(numeric_features)

scaled_numeric = X_train_preview[:, :n_numeric]

scale_check = pd.DataFrame({
    "Feature": numeric_features,
    "Mean After Scaling": scaled_numeric.mean(axis=0),
    "Std After Scaling": scaled_numeric.std(axis=0)
})

print("\n=== KIỂM TRA STANDARDSCALER ===")
display(scale_check.head(15).round(6))

print("\n=== TÊN ĐẶC TRƯNG SAU BIẾN ĐỔI ===")
display(
    pd.DataFrame({
        "Feature": feature_names
    })
)

=== KÍCH THƯỚC TRƯỚC TIỀN XỬ LÝ ===
Train: (82536, 82)
Validation: (17687, 82)

=== KÍCH THƯỚC SAU TIỀN XỬ LÝ ===
Train: (82536, 92)
Validation: (17687, 92)

Số đặc trưng sau tiền xử lý: 92

=== KIỂM TRA GIÁ TRỊ ===
Train có NaN/Inf: False
Validation có NaN/Inf: False

=== KIỂM TRA STANDARDSCALER ===


,Feature,Mean After Scaling,Std After Scaling
0,id.orig_p,-0.0,1.0
1,id.resp_p,-0.0,1.0
2,flow_duration,0.0,1.0
3,fwd_pkts_tot,-0.0,1.0
4,bwd_pkts_tot,-0.0,1.0
5,fwd_data_pkts_tot,-0.0,1.0
6,bwd_data_pkts_tot,-0.0,1.0
7,fwd_pkts_per_sec,0.0,1.0
8,bwd_pkts_per_sec,0.0,1.0
9,flow_pkts_per_sec,0.0,1.0



=== TÊN ĐẶC TRƯNG SAU BIẾN ĐỔI ===


,Feature
0,numeric__id.orig_p
1,numeric__id.resp_p
2,numeric__flow_duration
3,numeric__fwd_pkts_tot
4,numeric__bwd_pkts_tot
...,...
87,categorical__service_irc
88,categorical__service_mqtt
89,categorical__service_ntp
90,categorical__service_ssh


### Nhận xét: Kiểm tra kết quả tiền xử lý

**Kết quả cần kiểm tra:**

- Train và Validation có cùng số đặc trưng
  sau khi biến đổi.
- Các cột `proto` và `service` được mã hóa
  thành các đặc trưng nhị phân.
- Các đặc trưng số sau StandardScaler
  có trung bình gần 0 trên Train.
- Độ lệch chuẩn gần 1 với các đặc trưng
  có phương sai khác 0.
- Không phát hiện NaN hoặc Inf sau biến đổi.

**Kết luận:**

Bộ tiền xử lý hoạt động hợp lệ nếu vượt qua
toàn bộ kiểm tra.

Việc fit trên toàn bộ Train ở Cell này chỉ nhằm
kiểm thử kết quả biến đổi.

# 5. Lưu cấu hình và thống kê bộ tiền xử lý

In [5]:
import json

# Lưu danh sách đặc trưng
feature_table = pd.DataFrame({
    "Feature": feature_names
})

feature_table.to_csv(
    TABLE_DIR / "P01_feature_names.csv",
    index=False,
    encoding="utf-8-sig"
)

# Lưu kết quả kiểm tra scaler
scale_check.to_csv(
    TABLE_DIR / "P01_scaler_check.csv",
    index=False,
    encoding="utf-8-sig"
)

# Cấu hình tiền xử lý
metadata = {
    "experiment": "P01_MLP_Preprocessing_Baseline",
    "target": TARGET,
    "numeric_method": "StandardScaler",
    "categorical_method": "OneHotEncoder",
    "categorical_features": categorical_features,
    "numeric_feature_count": len(numeric_features),
    "input_feature_count": int(X_train.shape[1]),
    "output_feature_count": int(X_train_preview.shape[1]),
    "train_samples": int(len(X_train)),
    "validation_samples": int(len(X_valid)),
    "oversampling": False,
    "feature_selection": False,
    "test_used": False
}

with open(
    RESULT_DIR / "P01_preprocessing_info.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        metadata,
        f,
        indent=4,
        ensure_ascii=False
    )

print("=== KẾT QUẢ P01 ===")
display(pd.DataFrame([metadata]).T)

print("\nĐã lưu thông tin tiền xử lý tại:")
print(RESULT_DIR)

=== KẾT QUẢ P01 ===


,0
experiment,P01_MLP_Preprocessing_Baseline
target,Attack_type
numeric_method,StandardScaler
categorical_method,OneHotEncoder
categorical_features,"[proto, service]"
numeric_feature_count,80
input_feature_count,82
output_feature_count,92
train_samples,82536
validation_samples,17687



Đã lưu thông tin tiền xử lý tại:
D:\SGU26K2_MayHoc_DCT124C3_Nhom4_RT-IoT2022\final_IoT\sample_project\result\preprocessing\P01_MLP_Preprocessing_Baseline


### Tổng hợp kết quả P01 – MLP Preprocessing Baseline

**Phương pháp:**

- StandardScaler cho đặc trưng số.
- One-Hot Encoding cho proto và service.
- Không thực hiện Oversampling.
- Không thực hiện Feature Selection.
- Không xử lý ngoại lệ.

**Kết quả:**

Đã xây dựng và kiểm tra bộ tiền xử lý
cơ bản cho mô hình MLP.

Đã lưu danh sách đặc trưng sau mã hóa,
kết quả kiểm tra StandardScaler và
thông tin cấu hình tiền xử lý.

**Kết luận:**

Phương án này sẽ được dùng làm mốc để tiền xử lý
cho MLP Baseline.

Các phương pháp khác như RobustScaler, Log1p,
Correlation Filter và RandomOverSampler sẽ
được thực nghiệm riêng để đánh giá hiệu quả.